
# 応用問題 (最上級・発展): 物理情報ニューラルネット (PINN) で微分方程式を解く

## 目標

微分方程式

```
-u''(x) = f(x),   u(0) = u(1) = 0
```

を, 差分法 (B系) でも CG法 (G1) でもなく, **ニューラルネット (機械学習)** で解く。これが **物理情報ニューラルネット (PINN, Physics-Informed Neural Network)** の考え方である。**同じ問題を全く別の道具で解く**ことを体験する, 最上級の発展課題である。

PINN では, NN の出力 `u(x)` を解の候補とし, **PDE の残差そのものを損失**にして「解になるように」パラメータを学習する。

## 課題

1隠れ層・`tanh` のネットを使うと, `u` の微分は **解析的に書ける** (自動微分エンジン不要):

```
u(x)   = Σ_k a_k tanh(z_k),         z_k = w_k x + b_k
u'(x)  = Σ_k a_k w_k   sech^2(z_k),  sech^2 = 1 - tanh^2
u''(x) = Σ_k a_k w_k^2 (-2 tanh(z_k) sech^2(z_k))
```

製造解 (manufactured solution) として `u*(x) = sin(pi x)` (`u(0)=u(1)=0` を満たす) を選ぶと, `f(x) = -u*'' = pi^2 sin(pi x)`。

損失 (PDE 残差 + 境界ペナルティ):

```
loss = (1/M) Σ_i ( -u''(x_i) - f(x_i) )^2  +  lambda_bc ( u(0)^2 + u(1)^2 )
```

`M` 個のコロケーション点 `x_i ∈ (0,1)` で残差を測り, `lambda_bc ~ 10`。

パラメータ `{a_k, w_k, b_k}` (合計 `3H` 個) を勾配降下で学習する。勾配は**パラメータについての差分 (有限差分)** で求める:

```
grad_j = ( loss(p + eps·e_j) - loss(p - eps·e_j) ) / (2 eps)
```

この **`3H` 個の評価は互いに独立** (各 `j` で `loss` を 2 回呼ぶだけ) なので, **パラメータ番号 `j` で並列化**できる。`loss` はパラメータ配列の純関数 (各スレッドが自分のコピーを摂動) なのでスレッドセーフである。

- C++: `#pragma omp parallel for` を `for (j ...)` に付ける
- Fortran: `!$omp parallel do private(j,q,pp,lp,lm)` … `!$omp end parallel do`

これが `TODO` の並列化箇所である (パラメータについて embarrassingly parallel)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore pinn.cpp -o pinn.exe

# Fortran
nvfortran -fast -mp=multicore pinn.f90 -o pinn.exe
```

引数: 隠れユニット数 `H` (既定 16), コロケーション点数 `M` (既定 50), ステップ数 `steps` (既定 4000), 学習率 `lr` (既定 0.01)。

```
OMP_NUM_THREADS=4 ./pinn.exe 16 50 4000 0.01
```

## 期待される結果

```
step    0: loss=4.252154e+01
step 1000: loss=1.129228e-02
step 2000: loss=9.033110e-04
step 3000: loss=4.090927e-04
step 3999: loss=3.807349e-04
H=16, M=50, steps=4000, lr=0.01
final max|u - sin(pi x)| = 1.2837e-04
elapsed = ... sec
```

- 学習が進むと **損失が下がり**, 学習した `u(x)` が厳密解 `sin(pi x)` に近づく。最終的な **最大誤差は 0.05 を大きく下回る** (約 1e-4)。
- `OMP_NUM_THREADS` を増やすと `elapsed` が短くなる (台数効果)。結果 (誤差) は本質的に同じになる。
- 同じ `-u''=f` を, B系 (差分法) や G1 (CG法) とは全く別の道具 (機械学習) で解いたことを味わってほしい。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ pinn.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (パラメータ初期化用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* 物理情報ニューラルネット (PINN) で微分方程式
     -u''(x) = f(x),  u(0)=u(1)=0
   を「ニューラルネット」で解く。差分法(B系)・CG(G1)と同じ問題を機械学習で解く。

   1隠れ層・tanh のネットなので u の微分は解析的に書ける (autodiff 不要):
     u(x)   = Σ_k a_k tanh(z_k),         z_k = w_k x + b_k
     u'(x)  = Σ_k a_k w_k   sech^2(z_k),  sech^2 = 1 - tanh^2
     u''(x) = Σ_k a_k w_k^2 (-2 tanh(z_k) sech^2(z_k))

   厳密解 u*(x) = sin(pi x) (u(0)=u(1)=0 を満たす) を仕掛けると f(x) = pi^2 sin(pi x)。
   損失 = (1/M) Σ_i ( -u''(x_i) - f(x_i) )^2 + lambda_bc ( u(0)^2 + u(1)^2 )。
   パラメータ {a_k, w_k, b_k} (3H 個) を勾配降下で学習する。
   勾配はパラメータについての差分 (有限差分) で求める:
     grad_j = ( loss(p + eps e_j) - loss(p - eps e_j) ) / (2 eps)。
   この 3H 個の評価は互いに独立なので, パラメータ番号 j で並列化できる。 */

static const double PI = 3.14159265358979323846;

/* 損失をパラメータ配列 p (長さ 3H: a[0..H-1], w[H..2H-1], b[2H..3H-1]) の純関数として
   計算する (スレッドセーフ)。x は格子点 (長さ M)。 */
static double loss_fn(int H, const double * p, int M, const double * xs, double lambda_bc) {
  const double * a = p;
  const double * w = p + H;
  const double * b = p + 2*H;
  /* PDE 残差項 */
  double res = 0.0;
  for (int i = 0; i < M; i++) {
    double x = xs[i];
    double upp = 0.0;        /* u''(x) */
    for (int k = 0; k < H; k++) {
      double z = w[k]*x + b[k];
      double t = tanh(z);
      double s2 = 1.0 - t*t;            /* sech^2 */
      upp += a[k] * w[k]*w[k] * (-2.0*t*s2);
    }
    double f = PI*PI*sin(PI*x);         /* f = -u*'' = pi^2 sin(pi x) */
    double r = -upp - f;
    res += r*r;
  }
  res /= (double)M;
  /* 境界項: u(0), u(1) */
  double u0 = 0.0, u1 = 0.0;
  for (int k = 0; k < H; k++) {
    u0 += a[k] * tanh(b[k]);
    u1 += a[k] * tanh(w[k] + b[k]);
  }
  return res + lambda_bc * (u0*u0 + u1*u1);
}

int main(int argc, char ** argv) {
  int    H     = (argc > 1 ? atoi(argv[1]) : 16);     /* 隠れユニット数 */
  int    M     = (argc > 2 ? atoi(argv[2]) : 50);     /* コロケーション点数 */
  int    steps = (argc > 3 ? atoi(argv[3]) : 4000);   /* 勾配降下ステップ数 */
  double lr    = (argc > 4 ? atof(argv[4]) : 0.01);   /* 学習率 */
  double lambda_bc = 10.0;
  double eps = 1e-5;
  int P = 3*H;                                        /* パラメータ総数 */

  /* コロケーション点 x_i = (i+0.5)/M を (0,1) に等間隔配置 */
  double * xs = (double *)malloc(sizeof(double) * M);
  for (int i = 0; i < M; i++) xs[i] = (i + 0.5) / M;

  /* パラメータ p[3H] = {a, w, b} を小さな乱数で初期化 */
  double * p    = (double *)malloc(sizeof(double) * P);
  double * grad = (double *)malloc(sizeof(double) * P);
  for (int k = 0; k < H; k++) {
    p[k]       = (draw_rand01(k, 1) - 0.5) * 0.2;   /* a */
    p[H+k]     = (draw_rand01(k, 2) - 0.5) * 4.0;   /* w */
    p[2*H+k]   = (draw_rand01(k, 3) - 0.5) * 2.0;   /* b */
  }

  double t0 = omp_get_wtime();
  for (int it = 0; it < steps; it++) {
    /* パラメータについての差分勾配。各 j は独立 (loss は純関数なので各スレッドが
       自分のコピーを摂動して評価する)。 */
    // TODO: 3H 個のパラメータについての差分勾配ループを #pragma omp parallel for で並列化せよ (各反復で loss() を 2 回呼ぶ).
    for (int j = 0; j < P; j++) {
      double * pp = (double *)malloc(sizeof(double) * P);
      for (int q = 0; q < P; q++) pp[q] = p[q];
      pp[j] = p[j] + eps; double lp = loss_fn(H, pp, M, xs, lambda_bc);
      pp[j] = p[j] - eps; double lm = loss_fn(H, pp, M, xs, lambda_bc);
      grad[j] = (lp - lm) / (2.0 * eps);
      free(pp);
    }
    for (int j = 0; j < P; j++) p[j] -= lr * grad[j];

    if (it % 1000 == 0 || it == steps - 1) {
      double L = loss_fn(H, p, M, xs, lambda_bc);
      printf("step %4d: loss=%.6e\n", it, L);
    }
  }
  double elapsed = omp_get_wtime() - t0;

  /* 検算: 学習した u(x) を厳密解 sin(pi x) と比べる */
  const double * a = p; const double * w = p + H; const double * b = p + 2*H;
  double maxerr = 0.0;
  int NC = 101;
  for (int i = 0; i < NC; i++) {
    double x = (double)i / (NC - 1);
    double u = 0.0;
    for (int k = 0; k < H; k++) u += a[k] * tanh(w[k]*x + b[k]);
    double e = fabs(u - sin(PI*x));
    if (e > maxerr) maxerr = e;
  }
  printf("H=%d, M=%d, steps=%d, lr=%g\n", H, M, steps, lr);
  printf("final max|u - sin(pi x)| = %.4e\n", maxerr);
  printf("elapsed = %.3f sec\n", elapsed);
  free(xs); free(p); free(grad);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore pinn.cpp -o pinn_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./pinn_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ pinn.f90
module pinn_mod
  real(8), parameter :: PI = 3.14159265358979323846d0
contains
  ! 状態を持たない乱数 (パラメータ初期化用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! 損失をパラメータ配列 p (長さ 3H: a=p(1..H), w=p(H+1..2H), b=p(2H+1..3H)) の
  ! 純関数として計算する (スレッドセーフ)。xs は格子点 (長さ M)。
  function loss_fn(H, p, M, xs, lambda_bc) result(L)
    integer, intent(in) :: H, M
    real(8), intent(in) :: p(3*H), xs(M), lambda_bc
    real(8) :: L, res, upp, x, z, t, s2, f, r, u0, u1
    integer :: i, k
    res = 0.0d0
    do i = 1, M
       x = xs(i)
       upp = 0.0d0
       do k = 1, H
          z = p(H+k)*x + p(2*H+k)
          t = tanh(z)
          s2 = 1.0d0 - t*t
          upp = upp + p(k) * p(H+k)*p(H+k) * (-2.0d0*t*s2)
       end do
       f = PI*PI*sin(PI*x)
       r = -upp - f
       res = res + r*r
    end do
    res = res / real(M, 8)
    u0 = 0.0d0; u1 = 0.0d0
    do k = 1, H
       u0 = u0 + p(k) * tanh(p(2*H+k))
       u1 = u1 + p(k) * tanh(p(H+k) + p(2*H+k))
    end do
    L = res + lambda_bc * (u0*u0 + u1*u1)
  end function loss_fn
end module pinn_mod

! 物理情報ニューラルネット (PINN) で -u''(x)=f(x), u(0)=u(1)=0 を「NNで」解く。
! 1隠れ層・tanh なので u'' は解析的 (autodiff 不要)。厳密解 u*=sin(pi x), f=pi^2 sin(pi x)。
! パラメータについての差分勾配の 3H 個の評価を parallel do で分担する。
! 差分法(B系)・CG(G1)と同じ問題を全く別の道具(機械学習)で解く, 最上級の発展課題。
program pinn
  use pinn_mod
  use omp_lib
  character(len=32) :: arg
  integer :: H, M, steps, it, j, q, k, NP, NC, i
  real(8) :: lr, lambda_bc, eps, lp, lm, L, x, u, e, maxerr, t0, elapsed
  real(8), allocatable :: xs(:), p(:), grad(:), pp(:)

  H = 16; M = 50; steps = 4000; lr = 0.01d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) H
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) M
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) steps
  end if
  if (command_argument_count() >= 4) then
     call get_command_argument(4, arg); read (arg, *) lr
  end if
  lambda_bc = 10.0d0; eps = 1.0d-5; NP = 3*H

  ! コロケーション点 x_i = (i-0.5)/M を (0,1) に等間隔配置
  allocate(xs(M))
  do i = 1, M
     xs(i) = (real(i,8) - 0.5d0) / real(M, 8)
  end do

  ! パラメータ p(3H) = {a, w, b} を小さな乱数で初期化
  allocate(p(NP), grad(NP))
  do k = 1, H
     p(k)     = (draw_rand01(int(k-1,8), 1_8) - 0.5d0) * 0.2d0   ! a
     p(H+k)   = (draw_rand01(int(k-1,8), 2_8) - 0.5d0) * 4.0d0   ! w
     p(2*H+k) = (draw_rand01(int(k-1,8), 3_8) - 0.5d0) * 2.0d0   ! b
  end do

  t0 = omp_get_wtime()
  do it = 0, steps - 1
     ! パラメータについての差分勾配。各 j は独立 (loss は純関数なので各スレッドが
     ! 自分のコピー pp を摂動して評価する)。
     ! TODO: 3H 個のパラメータの差分勾配ループを !$omp parallel do private(j,q,pp,lp,lm) で並列化せよ (各反復で loss_fn を 2 回呼ぶ)。
     do j = 1, NP
        allocate(pp(NP))
        do q = 1, NP
           pp(q) = p(q)
        end do
        pp(j) = p(j) + eps; lp = loss_fn(H, pp, M, xs, lambda_bc)
        pp(j) = p(j) - eps; lm = loss_fn(H, pp, M, xs, lambda_bc)
        grad(j) = (lp - lm) / (2.0d0 * eps)
        deallocate(pp)
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     do j = 1, NP
        p(j) = p(j) - lr * grad(j)
     end do

     if (mod(it, 1000) == 0 .or. it == steps - 1) then
        L = loss_fn(H, p, M, xs, lambda_bc)
        print "(a,i4,a,es13.6)", "step ", it, ": loss=", L
     end if
  end do
  elapsed = omp_get_wtime() - t0

  ! 検算: 学習した u(x) を厳密解 sin(pi x) と比べる
  maxerr = 0.0d0; NC = 101
  do i = 0, NC - 1
     x = real(i, 8) / real(NC - 1, 8)
     u = 0.0d0
     do k = 1, H
        u = u + p(k) * tanh(p(H+k)*x + p(2*H+k))
     end do
     e = abs(u - sin(PI*x))
     if (e > maxerr) maxerr = e
  end do
  print "(a,i0,a,i0,a,i0,a,f0.4)", "H=", H, ", M=", M, ", steps=", steps, ", lr=", lr
  print "(a,es12.4)", "final max|u - sin(pi x)| = ", maxerr
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program pinn

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore pinn.f90 -o pinn_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./pinn_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:pinn.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:pinn.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:pinn.cpp}